# Monthly Portfolio Performance

This notebook calculates monthly portfolio performance from consecutive calendar month-end values. Portfolio values are evaluated on the last day of each month; when no price exists on that day, the portfolio service uses the latest available price on or before the month end.

For month `t`, performance is calculated as:

`((portfolio value at the end of t - net I/O during t) / portfolio value at the end of t-1) - 1`

In other words, the calculation uses the value at the end of the previous month as the starting value and the value at the end of the current month as the ending value. Net I/O is inflows minus outflows. Subtracting it removes deposits from performance and adds withdrawals back, so the result reflects portfolio appreciation rather than contributed capital.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root))

import pandas as pd
from IPython.display import display

from src.core.services import MonthlyPerformanceService

In [ ]:
monthly_performance = MonthlyPerformanceService().calculate()

performance_table = pd.DataFrame([
    {
        'month': result.month.strftime('%Y-%m'),
        'start_month_end': result.start_month_end,
        'end_month_end': result.end_month_end,
        'start_value': result.start_value,
        'inflow': result.inflow,
        'outflow': result.outflow,
        'net_io': result.net_io,
        'end_value': result.end_value,
        'performance': result.performance,
    }
    for result in monthly_performance
])

print(f'Months with performance: {len(performance_table)}')
display(performance_table)

In [ ]:
# The service returns Decimal values; convert only for chart rendering.
# The table above preserves the exact business values.

In [ ]:
import matplotlib.pyplot as plt

chart_table = performance_table.dropna(subset=['performance'])
plt.figure(figsize=(12, 5))
plt.bar(
    chart_table['month'],
    chart_table['performance'].astype(float) * 100,
    color=['#2a9d8f' if value >= 0 else '#e76f51' for value in chart_table['performance']],
)
plt.axhline(0, color='#333333', linewidth=0.8)
plt.title('Monthly Portfolio Performance')
plt.xlabel('Month')
plt.ylabel('Performance (%)')
plt.grid(axis='y', alpha=0.25)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()